In [38]:
import pandas as pd

df = pd.read_csv("learnit_courses.csv")

# Extract period from course name if it ends with e.g. "(Spring 2024)" or "(Autumn 2024)"
period_from_name = df["Course Name"].str.extract(r"\(((?:Spring|Autumn)\s+\d{4})\)$")[0]

course = pd.DataFrame()

# Use period from Course Name when present, otherwise use Semester
course["period"] = period_from_name.fillna(df["Semester"])

# Remove trailing "(Spring 2024)" / "(Autumn 2024)" from title
course["title"] = (
    df["Course Name"]
    .str.replace(r"\s+\((?:Spring|Autumn)\s+\d{4}\)$", "", regex=True)
    .str.replace(",", " -", regex=False)
    .str.strip()
)

# Keep language only if it is English or Danish
course["language"] = df["Language"].where(df["Language"].isin(["English", "Danish"]), "")

# Drop duplicates
course = course.drop_duplicates(subset=["period", "title","language"])

# Reset index and use it as ID starting from 0
course = course.reset_index(drop=True)
course.insert(0, "id", course.index)

course.to_csv("../CsvForDB/Course.csv", index=False)

In [39]:
import pandas as pd
import re
import unicodedata

# Load files
course_df = pd.read_csv("../CsvForDB/Course.csv")      # id, period, title, language
learnit_df = pd.read_csv("learnit_courses.csv")        # original export
person_df = pd.read_csv("../CsvForDB/person.csv")      # UUID, firstName, lastName, ...

def normalize_text(s):
    if pd.isna(s):
        return ""
    s = str(s)
    s = unicodedata.normalize("NFKC", s)
    s = s.replace("\u00a0", " ")
    s = s.replace("\u200b", "")
    s = s.replace("\u200c", "")
    s = s.replace("\u200d", "")
    s = s.replace("\ufeff", "")
    s = re.sub(r"\s+", " ", s)
    return s.strip().lower()

def extract_period(course_name, semester):
    """
    Use '(Spring 2024)' / '(Autumn 2024)' from Course Name if present.
    Otherwise fall back to Semester.
    """
    if pd.notna(course_name):
        m = re.search(r"\((Spring|Autumn)\s+\d{4}\)\s*$", str(course_name), flags=re.IGNORECASE)
        if m:
            return m.group(0).strip("()")
    return semester

def clean_title(s):
    s = normalize_text(s)
    s = re.sub(r"\s*\((spring|autumn)\s+\d{4}\)\s*$", "", s, flags=re.IGNORECASE)
    s = s.replace(",", " -")
    s = re.sub(r"\s+", " ", s)
    return s.strip()

# Normalize Course.csv
course_df["title_clean"] = course_df["title"].apply(clean_title)
course_df["period_clean"] = course_df["period"].apply(normalize_text)

# Normalize learnit_courses.csv
learnit_df["title_clean"] = learnit_df["Course Name"].apply(clean_title)
learnit_df["derived_period"] = learnit_df.apply(
    lambda r: extract_period(r["Course Name"], r["Semester"]),
    axis=1
)
learnit_df["period_clean"] = learnit_df["derived_period"].apply(normalize_text)
learnit_df["teacher_clean"] = learnit_df["Teacher"].apply(normalize_text)

# Normalize person.csv
person_df["full_name"] = (
    person_df["firstName"].fillna("").astype(str).str.strip() + " " +
    person_df["lastName"].fillna("").astype(str).str.strip()
)
person_df["full_name_clean"] = person_df["full_name"].apply(normalize_text)

# Build lookup dictionaries
course_lookup = dict(zip(
    zip(course_df["title_clean"], course_df["period_clean"]),
    course_df["id"]
))

person_lookup = dict(zip(
    person_df["full_name_clean"],
    person_df["UUID"]
))

# Assign courseId and personUUID directly onto LearnIT rows
learnit_df["courseId"] = learnit_df.apply(
    lambda r: course_lookup.get((r["title_clean"], r["period_clean"])),
    axis=1
)
learnit_df["personUUID"] = learnit_df["teacher_clean"].map(person_lookup)

# Build PersonTeachesCourse output
result = learnit_df[["personUUID", "courseId"]].dropna().drop_duplicates().copy()
result["courseId"] = result["courseId"].astype(int)

# Save
result.to_csv("../CsvForDB/PersonTeachesCourse.csv", index=False)

# Debug
print("Total learnit rows:", len(learnit_df))
print("Missing course:", learnit_df["courseId"].isna().sum())
print("Missing teacher:", learnit_df["personUUID"].isna().sum())
print("Matched rows:", learnit_df.dropna(subset=["personUUID", "courseId"]).shape[0])
print("Final unique pairs:", len(result))

print(result.head(20).to_string(index=False))

Total learnit rows: 1995
Missing course: 4
Missing teacher: 518
Matched rows: 1474
Final unique pairs: 1409
                          personUUID  courseId
1b3cff7f-1549-4998-b52e-0ba9093e1d01         0
8785f18a-1bf3-4c63-8d8b-d3e5d516ca40         1
1b3cff7f-1549-4998-b52e-0ba9093e1d01         1
1b3cff7f-1549-4998-b52e-0ba9093e1d01         2
5c50bbc9-ed3d-417f-ba4d-e484eaf536b3         3
5f3b7745-99be-4d1b-8ce1-a413325c2008         7
1b3cff7f-1549-4998-b52e-0ba9093e1d01         5
1b3cff7f-1549-4998-b52e-0ba9093e1d01         6
1b3cff7f-1549-4998-b52e-0ba9093e1d01         7
1b3cff7f-1549-4998-b52e-0ba9093e1d01         8
1b3cff7f-1549-4998-b52e-0ba9093e1d01         9
5c50bbc9-ed3d-417f-ba4d-e484eaf536b3        10
5c50bbc9-ed3d-417f-ba4d-e484eaf536b3        11
a22e85da-82e3-410d-8ef1-31407362ba19        13
1b3cff7f-1549-4998-b52e-0ba9093e1d01        13
1b3cff7f-1549-4998-b52e-0ba9093e1d01        14
1b3cff7f-1549-4998-b52e-0ba9093e1d01        15
53c45a58-4c3c-424c-80a2-c39e733fd7c3        16

In [40]:
missing_teachers = (
    learnit_df[learnit_df["personUUID"].isna()][["Teacher", "teacher_clean"]]
    .drop_duplicates()
    .sort_values("Teacher")
)

print("Unique missing teacher values:", len(missing_teachers))
print(missing_teachers.to_string(index=False))

Unique missing teacher values: 49
                                       Teacher                                  teacher_clean
                          Annelie Banks Berner                           annelie banks berner
                                   Aske Kammer                                    aske kammer
                      Bent Steenholt Kragelund                       bent steenholt kragelund
                               Charlene Putney                                charlene putney
                Christian Balslev van Randwijk                 christian balslev van randwijk
                            Christina Neumayer                             christina neumayer
                                    Dag Svanæs                                     dag svanæs
                                 Daniel Cermak                                  daniel cermak
                                   Development                                    development
                          